# ConvMixer Configuration Sweep
Train ConvMixer variants (dim/depth) and export the table: Model, Params, Val (%), Test (%), Time (s).


In [4]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Ensure imports work whether the notebook is launched from repo root or notebooks/.
if '__file__' in globals():
    _HERE = Path(__file__).resolve().parent
else:
    _HERE = Path.cwd()

PROJECT_ROOT = _HERE if (_HERE / 'src').exists() else _HERE.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError(f'Cannot locate project root from: {_HERE}')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.baseline_experiment import run_baseline_suite
from src.baseline_protocol import make_fair_train_config, build_output_dir

RUN_ID = 'convmixer_config_sweep_v1'
DATA_DIR = (PROJECT_ROOT / 'data/features/mel').resolve()
BASE_OUTPUT_DIR = (PROJECT_ROOT / 'data/models/convmixer_config_sweep').resolve()
SELECTED_MODELS = (
    'convmixer_64_8',
    'convmixer_128_8',
    'convmixer_256_8',
    'convmixer_256_12',
    'convmixer_512_12',
    'convmixer_512_16',
)
SELECTED_SEEDS = (43, 44)

output_dir = build_output_dir(base_output_dir=BASE_OUTPUT_DIR, run_id=RUN_ID)
output_dir.mkdir(parents=True, exist_ok=True)

config = make_fair_train_config(
    data_dir=DATA_DIR,
    base_output_dir=BASE_OUTPUT_DIR,
    run_id=RUN_ID,
    model_names=SELECTED_MODELS,
    seeds=SELECTED_SEEDS,
)
config


TrainConfig(data_dir='/home/anhcbt/extend/workspace/convmixer_model/data/features/mel', output_dir='/home/anhcbt/extend/workspace/convmixer_model/data/models/convmixer_config_sweep/convmixer_config_sweep_v1', run_id='convmixer_config_sweep_v1', data_version='convmixer_config_sweep_v1', model_names=('convmixer_64_8', 'convmixer_128_8', 'convmixer_256_8', 'convmixer_256_12', 'convmixer_512_12', 'convmixer_512_16'), seeds=(43, 44), split_seed=42, train_ratio=0.75, val_ratio=0.15, batch_size=32, num_workers=4, num_epochs=25, lr=0.001, weight_decay=0.0001, patience=8, min_delta=0.0, scheduler_factor=0.2, scheduler_patience=3, scheduler_min_lr=1e-06, ast_official_model_size='tiny224', ast_official_fstride=10, ast_official_tstride=10, ast_official_input_fdim=128, ast_official_input_tdim=32, ast_official_imagenet_pretrain=False, ast_official_audioset_pretrain=False, ast_official_verbose=True, ast_official_auto_input_shape=True, ast_official_auto_norm_from_train=True, ast_official_norm_mean=Non

In [5]:
runs_df, summary_df, history_store, metadata = run_baseline_suite(config)

runs_selected_path = output_dir / 'convmixer_runs_selected.csv'
summary_selected_path = output_dir / 'convmixer_summary_selected.csv'
runs_df.to_csv(runs_selected_path, index=False)
summary_df.to_csv(summary_selected_path, index=False)

print('Saved:', runs_selected_path)
print('Saved:', summary_selected_path)
print('Metadata:', metadata)


[START] model=convmixer_64_8 seed=43 params=81485 device=cpu lr=0.001 wd=0.0001


[convmixer_64_8|seed43] Epoch 1/25 | train_loss=2.1152 | val_loss=1.7187 | val_acc=39.55% | lr=0.001000->0.001000 | improved
[convmixer_64_8|seed43] Epoch 2/25 | train_loss=1.5305 | val_loss=1.0900 | val_acc=56.09% | lr=0.001000->0.001000 | improved
[convmixer_64_8|seed43] Epoch 3/25 | train_loss=1.1467 | val_loss=0.8404 | val_acc=67.07% | lr=0.001000->0.001000 | improved
[convmixer_64_8|seed43] Epoch 4/25 | train_loss=0.9361 | val_loss=0.7379 | val_acc=72.63% | lr=0.001000->0.001000 | improved
[convmixer_64_8|seed43] Epoch 5/25 | train_loss=0.7828 | val_loss=0.6015 | val_acc=76.09% | lr=0.001000->0.001000 | improved
[convmixer_64_8|seed43] Epoch 6/25 | train_loss=0.6593 | val_loss=0.5069 | val_acc=81.35% | lr=0.001000->0.001000 | improved
[convmixer_64_8|seed43] Epoch 7/25 | train_loss=0.5800 | val_loss=0.3923 | val_acc=87.52% | lr=0.001000->0.001000 | improved
[convmixer_64_8|seed43] Epoch 8/25 | train_loss=0.5420 | val_loss=0.3745 | val_acc=86.92% | lr=0.001000->0.001000 | no_improv

In [7]:
model_order = [
    'convmixer_64_8',
    'convmixer_128_8',
    'convmixer_256_8',
    'convmixer_256_12',
    'convmixer_512_12',
    'convmixer_512_16',
]
model_label = {
    'convmixer_64_8': '64/8',
    'convmixer_128_8': '128/8',
    'convmixer_256_8': '256/8',
    'convmixer_256_12': '256/12',
    'convmixer_512_12': '512/12',
    'convmixer_512_16': '512/16',
}

val_rows = []
for key, hist in history_store.items():
    if '_seed' not in key:
        continue
    model_name, seed_text = key.rsplit('_seed', 1)
    if model_name not in model_order:
        continue
    try:
        seed = int(seed_text)
    except ValueError:
        continue
    val_curve = hist.get('val_acc', [])
    best_val = float(max(val_curve)) if len(val_curve) else np.nan
    val_rows.append({'model': model_name, 'seed': seed, 'best_val_acc': best_val})

val_df = pd.DataFrame(val_rows)
if val_df.empty:
    raise ValueError('Cannot compute validation accuracy table from history_store.')

merged_df = runs_df.merge(val_df, on=['model', 'seed'], how='left')

table_raw = (
    merged_df[merged_df['model'].isin(model_order)]
    .groupby('model', as_index=False)
    .agg(
        params=('params', 'mean'),
        val_pct=('best_val_acc', 'mean'),
        test_pct=('test_accuracy', 'mean'),
        time_s=('train_seconds', 'mean'),
    )
    .set_index('model')
    .reindex(model_order)
    .reset_index()
)

table_df = pd.DataFrame({
    'Model': table_raw['model'].map(model_label),
    'Params': table_raw['params'].map(lambda v: f'{int(round(v)):,}'),
    'Val (%)': table_raw['val_pct'].map(lambda v: f'{v:.2f}'),
    'Test (%)': table_raw['test_pct'].map(lambda v: f'{v:.2f}'),
    'Time (s)': table_raw['time_s'].map(lambda v: f'{v:.1f}'),
})

table_csv = output_dir / 'convmixer_configuration_comparison_table.csv'
table_raw_csv = output_dir / 'convmixer_configuration_comparison_table_raw.csv'
table_tex = output_dir / 'convmixer_configuration_comparison_table.tex'

table_df.to_csv(table_csv, index=False)
table_raw.to_csv(table_raw_csv, index=False)
table_df.to_latex(table_tex, index=False, escape=False)

print('Saved:', table_csv)
print('Saved:', table_raw_csv)
print('Saved:', table_tex)
table_df


Saved: /home/anhcbt/extend/workspace/convmixer_model/data/models/convmixer_config_sweep/convmixer_config_sweep_v1/convmixer_configuration_comparison_table.csv
Saved: /home/anhcbt/extend/workspace/convmixer_model/data/models/convmixer_config_sweep/convmixer_config_sweep_v1/convmixer_configuration_comparison_table_raw.csv
Saved: /home/anhcbt/extend/workspace/convmixer_model/data/models/convmixer_config_sweep/convmixer_config_sweep_v1/convmixer_configuration_comparison_table.tex


,Model,Params,Val (%),Test (%),Time (s)
0,64/8,"81,485",94.74,95.48,135.5
1,128/8,"228,493",96.17,96.15,212.0
2,256/8,"719,117",96.39,96.70,329.6
3,256/12,"1,070,349",96.62,96.26,576.7
4,512/12,"3,713,549",96.24,96.92,1218.7
5,512/16,"4,940,301",95.94,96.37,1610.7
